# Persian isolated letter recognizer

Run the fine-tuned YOLO11n model on an uploaded image in Google Colab. The model was trained on synthetic isolated typography. Camera photos and natural handwriting have not been validated. See `RESULTS.md` and `ATTRIBUTION.md` in the release package for measured results, limitations, and credits.

Place `yolo11n.pt` in the same Google Drive folder as this notebook, then run the cells.

In [ ]:
%pip -q install ultralytics==8.4.172
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json, torch
from ultralytics import YOLO
from google.colab import files
from IPython.display import display
from PIL import Image

matches = list(Path("/content/drive/MyDrive").rglob("yolo11n.pt"))
if not matches:
    raise FileNotFoundError("Upload yolo11n.pt to My Drive, then rerun.")
MODEL_PATH = matches[0]
model = YOLO(str(MODEL_PATH))
print("Loaded:", MODEL_PATH)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
labels_path = MODEL_PATH.parent / "labels.json"
labels = json.loads(labels_path.read_text()) if labels_path.exists() else None
policy = {"candidate_floor": 0.03, "acceptance_threshold": 0.8, "nms_iou": 0.4}

In [ ]:
uploaded = files.upload()
for filename in uploaded:
    result = model.predict(filename, imgsz=640, rect=False, conf=policy["candidate_floor"],
                           iou=policy["nms_iou"], agnostic_nms=True, max_det=32,
                           device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
    image = Image.open(filename).convert("RGB")
    display(image)
    boxes = result.boxes
    candidates = [(int(c), float(s)) for c, s in zip(boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist())]
    if len(candidates) == 1 and candidates[0][1] >= policy["acceptance_threshold"]:
        class_id, confidence = candidates[0]
        letter = labels[class_id]["letter"] if labels else str(class_id)
        transliteration = labels[class_id]["transliteration"] if labels else ""
        print(f"Accepted: {letter} ({transliteration}), confidence {confidence:.1%}")
    else:
        print(f"Retake or upload one clear isolated letter. Candidates: {candidates}")